# Ton premier projet Data & ML en Python

**Question :** peut-on estimer la consommation d'une voiture à partir de ses caractéristiques ?

On va suivre, dans l'ordre, les étapes de la formation :

1. **Collecte** — charger les données
2. **Exploration** — premier examen
3. **Nettoyage** — valeurs manquantes, faux doublons
4. **Exploration et prétraitement** — explorer, puis convertir
5. **Modélisation** — entraîner le modèle
6. **Interprétation** — évaluer et interpréter

*Données : Auto MPG, UCI Machine Learning Repository — Quinlan, R. (1993),
https://doi.org/10.24432/C5859H — licence CC BY 4.0.*

## 1. Collecte — charger les données

Auto MPG contient **398 voitures** commercialisées entre 1970 et 1982.
**Une ligne représente une voiture.** Les colonnes décrivent son poids, sa puissance,
sa cylindrée, son année et sa consommation.

Notre objectif : prédire une **quantité numérique continue**, la consommation.
C'est donc un problème de **régression**.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/formation-ia-data/premier-projet-data-ml/v1/data/mpg.csv"
df = pd.read_csv(URL)

## 2. Exploration — premier examen

In [ ]:
df.shape

In [ ]:
df.head(3)

In [ ]:
df.info()

In [ ]:
df.isna().sum()

Une vue réduite pour lire les variables une à une :

In [ ]:
df[["name", "mpg", "weight", "horsepower", "model_year"]].head()

## 3. Nettoyage

### Les valeurs manquantes

On ne supprime pas une ligne parce qu'elle contient une valeur manquante.
On regarde d'abord si cette valeur est **nécessaire pour notre objectif**.

In [ ]:
df[df["horsepower"].isna()]

Ces six voitures ont toutes un poids et une consommation valides : seule la puissance
manque, et notre modèle ne l'utilisera pas. **On les garde toutes.**

### Le piège des faux doublons

In [ ]:
print(df["name"].duplicated().sum())

In [ ]:
df[df["name"] == "ford pinto"]

In [ ]:
print(df.duplicated().sum())

Des noms répétés, mais **aucune ligne réellement dupliquée** : c'est le même modèle,
vendu plusieurs années de suite, avec un poids et une consommation différents.
Supprimer les « doublons » sur le nom détruirait des données parfaitement valides.

## 4. Exploration et prétraitement

### 4a. Explorer les données telles quelles

Variable indépendante : le poids. Variable dépendante : la consommation.

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df["weight"], df["mpg"], alpha=0.6)
plt.xlabel("Poids (livres)")
plt.ylabel("Consommation (miles per gallon)")
plt.show()

Deux constats :
- la relation est **décroissante**, et **courbe** plutôt que droite ;
- la variable dépendante est **à l'envers** : un `mpg` élevé signifie une **faible** consommation.

### 4b. Prétraiter : passer en unités métriques

In [ ]:
df["conso_l_100km"] = 235.214583 / df["mpg"]      # une inversion, pas une multiplication
df["poids_kg"]      = df["weight"] * 0.45359237

### 4c. Ré-explorer : le nuage est-il redressé ?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(df["poids_kg"], df["mpg"], alpha=0.6)
axes[0].set(xlabel="Poids (kg)", ylabel="mpg", title="Avant prétraitement — une courbe")
axes[1].scatter(df["poids_kg"], df["conso_l_100km"], alpha=0.6)
axes[1].set(xlabel="Poids (kg)", ylabel="L/100 km", title="Après prétraitement — une droite")
plt.show()

### Les corrélations

In [ ]:
cols = ["conso_l_100km", "poids_kg", "horsepower", "displacement",
        "cylinders", "acceleration", "model_year"]
df[cols].corr().round(2)

Le poids, la cylindrée et le nombre de cylindres sont **fortement corrélés entre eux**.
Ajouter toutes ces variables ne rendrait pas le modèle automatiquement meilleur :
cela rendrait ses coefficients instables et difficiles à interpréter. On garde **une seule variable indépendante : le poids.**

## 5. Modélisation — entraîner le modèle

On sépare les données en un **ensemble d'apprentissage** (80 %) et un **ensemble de test** (20 %).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X = df[["poids_kg"]]
y = df["conso_l_100km"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
print(len(X_train), "voitures pour l'apprentissage")
print(len(X_test), "voitures pour le test")

In [ ]:
modele = LinearRegression()
modele.fit(X_train, y_train)

Le modèle a calculé la droite **Y = AX + B** :
- `coef_` est **A**, la pente ;
- `intercept_` est **B**, l'ordonnée à l'origine.

In [ ]:
print("A (pente)                  :", modele.coef_[0])
print("B (ordonnée à l'origine)   :", modele.intercept_)
print("Pour 100 kg de plus        :", round(modele.coef_[0] * 100, 2), "L/100 km")

## 6. Interprétation — évaluer le modèle

On évalue sur l'**ensemble de test** : des voitures que le modèle n'a **jamais vues**.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = modele.predict(X_test)

print("Erreur absolue moyenne       :", round(mean_absolute_error(y_test, y_pred), 3), "L/100 km")
print("Coefficient de détermination :", round(r2_score(y_test, y_pred), 3))

In [ ]:
print("R² sur l'ensemble d'apprentissage :", round(modele.score(X_train, y_train), 3))
print("R² sur l'ensemble de test         :", round(modele.score(X_test, y_test), 3))

### Valeurs prédites face aux valeurs réelles observées

In [ ]:
resultats = pd.DataFrame({
    "voiture": df.loc[X_test.index, "name"],
    "poids_kg": X_test["poids_kg"].round(0),
    "reelle": y_test.round(2),
    "predite": y_pred.round(2),
})
resultats["erreur"] = (resultats["reelle"] - resultats["predite"]).abs().round(2)
resultats.head()

Les plus grosses erreurs :

In [ ]:
resultats.sort_values("erreur", ascending=False).head(5)

### La droite du modèle

In [ ]:
plt.scatter(X_test["poids_kg"], y_test, alpha=0.6, label="valeurs réelles observées")
plt.plot(X_test["poids_kg"], y_pred, color="red", label="droite du modèle")
plt.xlabel("Poids (kg)")
plt.ylabel("Consommation (L/100 km)")
plt.legend()
plt.show()

### Attention à l'extrapolation

Le modèle n'a vu que des voitures de **732 à 2 331 kg**. Hors de cette plage,
ses estimations n'ont plus de sens :

In [ ]:
exemples = pd.DataFrame({"poids_kg": [0, 1000, 1500, 2000]})
exemples["conso_estimee"] = modele.predict(exemples).round(2)
exemples

Une voiture de 0 kg consommerait moins de zéro litre : absurde.
Le modèle décrit une **association** dans un échantillon de voitures de 1970 à 1982.
Il ne prouve pas que le poids est la seule cause, et il ne dit rien des voitures d'aujourd'hui.

---
### Versions utilisées pour la vidéo

Si le notebook ne fonctionne plus un jour, cette cellule indique avec quelles versions
la vidéo a été tournée.

In [ ]:
import pandas, sklearn, matplotlib, sys
print(sys.version)
print("pandas", pandas.__version__, "| sklearn", sklearn.__version__,
      "| matplotlib", matplotlib.__version__)